<a href="https://colab.research.google.com/github/Prathambiradr12345/Ai-Agents/blob/main/Tool_Calling_in_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
import os
os.environ["MISTRAL_API_KEY"]="fc5CTVhYgslWCUju8NBILRAN2Zppn3EB"

In [61]:
!pip install -q langchain-mistralai langchain-core requests

In [62]:
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [63]:
#tool Create
@tool
def multiply(a:int,b:int) -> int:
  """Multiplies two integers a and b."""
  return a * b

In [64]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [65]:
multiply.name

'multiply'

In [66]:
multiply.description

'Multiplies two integers a and b.'

In [67]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [68]:
#tool Binding

In [69]:
llm=ChatMistralAI()

In [70]:
llmtools=llm.bind_tools([multiply])

In [71]:
llmtools

_ChatModelBinding(bound=ChatMistralAI(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.6'}}, client=<httpx.Client object at 0x7a51f2bd4a40>, async_client=<httpx.AsyncClient object at 0x7a51bffdd280>, mistral_api_key=SecretStr('**********'), endpoint='https://api.mistral.ai/v1', model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'multiply', 'description': 'Multiplies two integers a and b.', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [72]:
re=llmtools.invoke("What is 3 multiply 4").tool_calls[0]

In [73]:
re

{'name': 'multiply',
 'args': {'a': 3, 'b': 4},
 'id': 'mk65o0SnB',
 'type': 'tool_call'}

In [74]:
multiply.invoke({'a':3, 'b':4})

12

In [75]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [76]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [77]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1781481601,
 'time_last_update_utc': 'Mon, 15 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1781568001,
 'time_next_update_utc': 'Tue, 16 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.1765}

In [78]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [79]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [80]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [81]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [82]:
ai_message = llm_with_tools.invoke(messages)

In [83]:
messages.append(ai_message)

In [84]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '0cPYMWEDg',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '9TahHxMA6',
  'type': 'tool_call'}]

In [85]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)


In [86]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0cPYMWEDg', 'type': 'function', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "INR", "target_currency": "USD"}'}, 'index': 0}, {'id': '9TahHxMA6', 'type': 'function', 'function': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}, 'index': 1}]}, response_metadata={'token_usage': {'prompt_tokens': 200, 'total_tokens': 234, 'completion_tokens': 34, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ecc50-fa6b-7d03-9e75-f71f5d5ff9f9-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '0cPYMWEDg', 'type': 'tool_call'}, {'na

In [87]:
llm_with_tools.invoke(messages).content

'The conversion factor between **INR** and **USD** is **0.01051** (1 INR = 0.01051 USD).\n\nBased on this conversion rate, **10 INR** is equivalent to **0.1051 USD**.'

In [91]:
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# Initialize LLM
llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key="fc5CTVhYgslWCUju8NBILRAN2Zppn3EB"
)

# Example tools
@tool
def get_conversion_factor(currency: str) -> float:
    """Get conversion factor for a currency."""
    rates = {"INR": 83.0, "USD": 1.0}
    return rates.get(currency.upper(), 1.0)

@tool
def convert(amount: float, factor: float) -> float:
    """Convert amount using factor."""
    return amount * factor

# Create agent
agent = create_agent(
    model=llm,
    tools=[get_conversion_factor, convert]
)

# Invoke
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Convert 10 USD to INR"
            }
        ]
    }
)

print(response)

{'messages': [HumanMessage(content='Convert 10 USD to INR', additional_kwargs={}, response_metadata={}, id='2f1bca9e-4f34-49f1-88b5-d2e0278844a3'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'awSkHutbS', 'type': 'function', 'function': {'name': 'get_conversion_factor', 'arguments': '{"currency": "INR"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 149, 'total_tokens': 163, 'completion_tokens': 14, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ecc53-4d21-75a2-9f55-ee3179e41ba9-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'currency': 'INR'}, 'id': 'awSkHutbS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_tokens': 14, 'total_tokens': 163}), ToolMessage(content='83.0', name='get_conversion_factor', id='83013e23-d35c-4b85-aba2-ce7d3a2f85b9'